# Simple approach to RAG with LangChain and Docker Model Runner using OpenAI-compliant code

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) setup using LangChain and OpenAI.

### Load env vars

In [1]:
import os

from dotenv import load_dotenv
from huggingface_hub import login
from pyspark.sql import SparkSession

# Load environment variables from .env file
load_dotenv()

# Login to Hugging Face
# login(token=os.getenv("HF_TOKEN"))

# Set environment variables
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")

# Initialize Spark session
spark = SparkSession.builder.appName("pyspark-docs-inspect").getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/28 10:45:32 WARN Utils: Your hostname, Samuel-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/28 10:45:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/28 10:45:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.0


## Indexing

### Document Loader


In [2]:
import sys
from pathlib import Path

# Data directory and file paths
DATA_DIR = Path("./data")
DOC_FILE = DATA_DIR / "pyspark_docs.jsonl"

# If ./data/pyspark_docs.jsonl is missing, run sparksql_docs_web_scraping.py to create it.
if DOC_FILE.exists():
    print(f"Found dataset: {DOC_FILE}")
else:
    print(f"Dataset not found: {DOC_FILE}")
    print("Running sparksql_docs_web_scraping.build_jsonl() to create the dataset...")

    # Create data directory if it doesn't exist
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    try:
        import sparksql_docs_web_scraping

        sparksql_docs_web_scraping.build_jsonl(limit=None, out_path=DOC_FILE)
        print(f"Dataset created at: {DOC_FILE}")
    except ImportError:
        print(
            "Error: sparksql_docs_web_scraping module not found. Please ensure it is available."
        )
        sys.exit(1)

Found dataset: data/pyspark_docs.jsonl


In [3]:
# Load the data
df = spark.read.option("multiline", False).json(str(DOC_FILE))

# Display the number of rows and schema
print("Loaded docs:", df.count())
df.printSchema()

# Display the first few rows
df.show(5, truncate=False)

Loaded docs: 890
root
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Basic Doc spliter


In [4]:
import tiktoken
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Define the directory for Chroma vector store
PERSIST_DIR = "./chroma_pyspark_basic"
print("Chroma collection persisted at:", PERSIST_DIR)


add_docs = not os.path.exists(PERSIST_DIR)
print("Adding documents to vector store:", add_docs)

# Get chunk_size in tokens
enc = tiktoken.get_encoding("cl100k_base")  # good default for most LLMs


def tiktoken_len(s: str) -> int:
    return len(enc.encode(s))


# Define the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=110,
    chunk_overlap=30,
    length_function=tiktoken_len,
    separators=[
        "\n```python",
        "\n```",
        "```",  # code fences first
        "\nParameters\n",
        "\nReturns\n",
        "\nNotes\n",
        "\nExamples\n",
        "\nMethods\n",
        "\n\n",
        "\n",
        " ",
        "",
    ],
)


# Function to extract symbol from URL
def symbol_from_url(url: str) -> str:
    return url.rsplit("/", 1)[-1].replace(".html", "")


if add_docs:
    page_docs = []
    for row in df.collect():
        page_docs.append(
            Document(
                page_content=row["title"] + " " + row["text"],
                metadata={
                    "source": row["url"],
                    "symbol": symbol_from_url(row["url"]),
                    "lib": "pyspark",
                    "version": "latest",
                    "chunker": "lc_recursive_token_v1",
                },
            )
        )

    docs = text_splitter.split_documents(page_docs)

    # Display the number of chunks created
    print(f"Built {len(docs):,} chunks.")

Chroma collection persisted at: ./chroma_pyspark_basic
Adding documents to vector store: False


### Embedding class
Docker model runner is compatible with OpenAI API, because of that the embedding class is built like this.


In [30]:
from typing import List

from langchain_core.embeddings import Embeddings
from openai import OpenAI

# Set up OpenAI client
client = OpenAI(api_key="not-needed", base_url=os.getenv("LLM_BASE_URL"))
EMBED_MODEL = "ai/mxbai-embed-large"


# Chroma expects an object with .embed_documents(list[str]) and .embed_query(str)
# This class wraps the OpenAI client to batch calls to /embeddings on the local model
class OpenAICompatEmbeddings(Embeddings):
    """
    OpenAI-compatible embeddings class for LangChain.
    This class wraps the OpenAI client to provide embeddings functionality.

    Args:
        client (OpenAI): The OpenAI client instance.
        model (str): The model name to use for embeddings. (local model)
        batch_size (int): The number of texts to process in a single batch.
    Returns:
        List[List[float]]: A list of embeddings for the input texts.
        List[float]: A single embedding for the input query text.
    """

    def __init__(self, client: OpenAI, model: str, batch_size: int = 64):
        self.client = client
        self.model = model
        self.batch_size = batch_size

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        vecs: List[List[float]] = []
        # Simple batching
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            resp = self.client.embeddings.create(model=self.model, input=batch)
            # OpenAI SDK returns objects with .data[i].embedding
            vecs.extend([d.embedding for d in resp.data])
        return vecs

    def embed_query(self, text: str) -> List[float]:
        resp = self.client.embeddings.create(model=self.model, input=[text])
        return resp.data[0].embedding


embedding = OpenAICompatEmbeddings(client, EMBED_MODEL, batch_size=64)

### Build a Chroma vectorstore


In [ ]:
import hashlib

from langchain_chroma import Chroma

# Build the Chroma vectorstore from the documents
vector_store = Chroma(
    collection_name="pyspark_basic_char_v1",
    persist_directory=PERSIST_DIR,  # Persist across sessions
    embedding_function=embedding,
)

if add_docs:
    print("Adding documents to the vector store...")

    BATCH = 500  # Number of chunks to process in each batch

    # Function to create unique IDs for each document
    def make_id(doc: Document, global_idx: int) -> str:
        src = doc.metadata.get("source", "")
        # Add global_idx to ensure uniqueness
        return hashlib.md5(f"{src}|{global_idx}".encode("utf-8")).hexdigest()

    for start in range(0, len(docs), BATCH):
        end = min(len(docs), start + BATCH)
        batch_docs = docs[start:end]
        ids = [make_id(d, start + i) for i, d in enumerate(batch_docs)]
        vector_store.add_documents(batch_docs, ids=ids)
        print(f"Upserted {end}/{len(docs)}")

    print("Done adding documents.")
else:
    print("Reusing existing persisted collection; not adding docs.")


Reusing existing persisted collection; not adding docs.


#### Query the vectorstore directly


##### Similarity search
Performing a simple similarity search

In [33]:
query = "What is the difference between DataFrame and Dataset in PySpark?"

results = vector_store.similarity_search(query, k=2)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* pyspark.sql.DataFrame# pyspark.sql.DataFrame
class pyspark.sql.DataFrame(jdf, sql_ctx)
A distributed collection of data grouped into named columns.
New in version 1.3.0.
Changed in version 3.4.0: Supports Spark Connect.
Notes
A DataFrame should only be created as described above. It should not be directly created via using the constructor.
Examples
A DataFrame is equivalent to a relational table in Spark SQL, and can be created using various functions in SparkSession : [{'source': 'https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html', 'lib': 'pyspark', 'chunker': 'lc_recursive_token_v1', 'symbol': 'pyspark.sql.DataFrame', 'version': 'latest'}]
* pyspark.sql.DataFrameStatFunctions# pyspark.sql.DataFrameStatFunctions
class pyspark.sql.DataFrameStatFunctions(df)
Functionality for statistic functions with DataFrame .
New in version 1.4.0.
Changed in version 3.4.0: Supports Spark Connect. [{'lib': 'pyspark', 'chunker': 'lc_recursive_token_v

##### Similarity search with score

In [34]:
results = vector_store.similarity_search_with_score(query, k=2)

for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.445601] pyspark.sql.DataFrame# pyspark.sql.DataFrame
class pyspark.sql.DataFrame(jdf, sql_ctx)
A distributed collection of data grouped into named columns.
New in version 1.3.0.
Changed in version 3.4.0: Supports Spark Connect.
Notes
A DataFrame should only be created as described above. It should not be directly created via using the constructor.
Examples
A DataFrame is equivalent to a relational table in Spark SQL, and can be created using various functions in SparkSession : [{'version': 'latest', 'source': 'https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html', 'symbol': 'pyspark.sql.DataFrame', 'chunker': 'lc_recursive_token_v1', 'lib': 'pyspark'}]
* [SIM=0.490087] pyspark.sql.DataFrameStatFunctions# pyspark.sql.DataFrameStatFunctions
class pyspark.sql.DataFrameStatFunctions(df)
Functionality for statistic functions with DataFrame .
New in version 1.4.0.
Changed in version 3.4.0: Supports Spark Connect. [{'lib': 'pyspark', 'c

##### Search with vector

In [35]:
results = vector_store.similarity_search_by_vector(
    embedding=embedding.embed_query(query), k=2
)

for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* pyspark.sql.DataFrame# pyspark.sql.DataFrame
class pyspark.sql.DataFrame(jdf, sql_ctx)
A distributed collection of data grouped into named columns.
New in version 1.3.0.
Changed in version 3.4.0: Supports Spark Connect.
Notes
A DataFrame should only be created as described above. It should not be directly created via using the constructor.
Examples
A DataFrame is equivalent to a relational table in Spark SQL, and can be created using various functions in SparkSession : [{'lib': 'pyspark', 'source': 'https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html', 'chunker': 'lc_recursive_token_v1', 'symbol': 'pyspark.sql.DataFrame', 'version': 'latest'}]
* pyspark.sql.DataFrameStatFunctions# pyspark.sql.DataFrameStatFunctions
class pyspark.sql.DataFrameStatFunctions(df)
Functionality for statistic functions with DataFrame .
New in version 1.4.0.
Changed in version 3.4.0: Supports Spark Connect. [{'lib': 'pyspark', 'version': 'latest', 'source': '

##### MMR search

In [36]:
results = vector_store.max_marginal_relevance_search(query, k=2, fetch_k=20)

for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* pyspark.sql.DataFrame# pyspark.sql.DataFrame
class pyspark.sql.DataFrame(jdf, sql_ctx)
A distributed collection of data grouped into named columns.
New in version 1.3.0.
Changed in version 3.4.0: Supports Spark Connect.
Notes
A DataFrame should only be created as described above. It should not be directly created via using the constructor.
Examples
A DataFrame is equivalent to a relational table in Spark SQL, and can be created using various functions in SparkSession : [{'source': 'https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html', 'symbol': 'pyspark.sql.DataFrame', 'lib': 'pyspark', 'version': 'latest', 'chunker': 'lc_recursive_token_v1'}]
* ```python
>>> from pyspark.sql.functions import col
>>> dataset = spark.range(0, 100).select((col("id") % 3).alias("key"))
>>> sampled = dataset.sampleBy("key", fractions={0: 0.1, 1: 0.2}, seed=0)
>>> sampled.groupBy("key").count().orderBy("key").show()
+---+-----+
|key|count|
+---+-----+
| 0| 

## Retriver

In [37]:
# Create a retriever from the vectorstore
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [38]:
retrieved_docs = retriever.invoke(query)
for doc in retrieved_docs:
    print(f"* {doc.page_content} [{doc.metadata}]")

* pyspark.sql.DataFrame# pyspark.sql.DataFrame
class pyspark.sql.DataFrame(jdf, sql_ctx)
A distributed collection of data grouped into named columns.
New in version 1.3.0.
Changed in version 3.4.0: Supports Spark Connect.
Notes
A DataFrame should only be created as described above. It should not be directly created via using the constructor.
Examples
A DataFrame is equivalent to a relational table in Spark SQL, and can be created using various functions in SparkSession : [{'version': 'latest', 'symbol': 'pyspark.sql.DataFrame', 'source': 'https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html', 'lib': 'pyspark', 'chunker': 'lc_recursive_token_v1'}]
* pyspark.sql.DataFrameStatFunctions# pyspark.sql.DataFrameStatFunctions
class pyspark.sql.DataFrameStatFunctions(df)
Functionality for statistic functions with DataFrame .
New in version 1.4.0.
Changed in version 3.4.0: Supports Spark Connect. [{'symbol': 'pyspark.sql.DataFrameStatFunctions', 'c

## Generation

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="ai/llama3.1", base_url=os.getenv("LLM_BASE_URL"), api_key="not-needed"
)


SYSTEM_PROMPT = (
    "You are a strict PySpark assistant.\n"
    "Refuse or say “I don’t know” for anything not about PySpark APIs, behavior, or usage.\n"
    "Do NOT rewrite or reinterpret off-topic questions into PySpark topics.\n"
    "If the provided context is empty or clearly unrelated to the user question, reply:\n"
    "“I’m tuned to PySpark only. Please ask a PySpark question.”\n"
    "Always include a SOURCES section with the URLs you actually used."
)

USER_TEMPLATE = """
QUESTION: {question}
=========

{context}
=========

FINAL ANSWER: <your answer here>

SOURCES: <your sources here>
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("user", USER_TEMPLATE),
    ]
)


def format_docs(docs):
    parts = []
    for d in docs:
        src = d.metadata.get("source", "")
        parts.append(f"{d.page_content}\n(SOURCE: {src})")
    return "\n\n".join(parts)


rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [47]:
rag_chain.invoke(query)

'The main difference between a DataFrame and a Dataset in PySpark is the type of data they can hold and the level of type safety they provide.\n\nA `DataFrame` is a distributed collection of data grouped into named columns, which is equivalent to a relational table in Spark SQL. It is more like an old-school RDD and can hold any type of data, including objects that do not have a fixed schema.\n\nOn the other hand, a `Dataset` is a distributed collection of data that is strongly typed and has a fixed schema, similar to a Java or Python list. It is more like a typed RDD and provides better type safety and performance.\n\nIn general, `Datasets` are recommended over `DataFrames` when you have a fixed schema, as they provide better performance and type safety. However, `DataFrames` are still useful when you have a dynamic schema or need to work with unstructured data.\n\nHere\'s an example of the difference between the two:\n\n```python\nfrom pyspark.sql import SparkSession\n\n# Create a Da

In [48]:
rag_chain.invoke("How to create a DataFrame in PySpark?")

"To create a DataFrame in PySpark from a pandas DataFrame, you can use the `createDataFrame()` method with the `pandas` option.\n\n```python\nfrom pyspark.sql import SparkSession\nfrom pyspark.sql.functions import col\n\n# Create a pandas DataFrame\nimport pandas as pd\ndf_pandas = pd.DataFrame({\n    'name': ['Alice', 'Bob', 'Tom'],\n    'age': [10, 5, 7],\n    'height': [80, None, None]\n})\n\n# Create a PySpark DataFrame from the pandas DataFrame\ndf = spark.createDataFrame(df_pandas)\n\n# Show the DataFrame\ndf.show()\n```\n\nThis will create a PySpark DataFrame with the same structure and data as the pandas DataFrame.\n\n(SOURCES: \n\nhttps://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.createDataFrame.html\nhttps://pandas.pydata.org/docs/)"

In [49]:
rag_chain.invoke("Is pandas better than PySpark?")

'I don\'t know. This question is not about PySpark usage or behavior. Pandas and PySpark are both used for data manipulation and analysis, but the question of which one is "better" is subjective and depends on the specific use case. \n\nI’m tuned to PySpark only. Please ask a PySpark question. \n\nSOURCES: \n\n- https://spark.apache.org/docs/latest/api/python/\n- https://pandas.pydata.org/docs/'

In [50]:
rag_chain.invoke("How can I create an AI chatbot?")

"I don't know how to create an AI chatbot using the provided PySpark context. The information you've shared seems to be about PySpark DataFrame operations, including creating a map, joining dataframes, and using hints for physical plan optimization. Creating an AI chatbot is not related to these topics.\n\nI'm tuned to PySpark only. Please ask a PySpark question.\n\nSOURCES: \n\n* https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.create_map.html \n* https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html \n* https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.pivot.html"